In [1]:
from pathlib import Path
import json
import pandas as pd

RUN_ID = "ablation_3seed_alpha1_20260518_171506"
ALPHA = "1.000"
ROOT = Path("/mnt/data/nvth2.data/experiments")
DATASETS = ["cancer", "immune", "light"]
METHODS = {"shuffle": "Shuffle", "random_lr": "Random", "metacell": "Metacell"}
CELLBRIDGE_ARTIFACTS = {
    "cancer": ROOT / "cancer/align_sweep/2025-09-21/22-27-38/artifacts",
    "immune": ROOT / "immune/align_sweep/2025-09-21/21-34-03/artifacts",
    "light": ROOT / "light/align_sweep/2025-09-21/19-04-26/artifacts",
}
SEEDS = [42, 43, 44]

records = []
for dataset in DATASETS:
    for slug, method in METHODS.items():
        for seed in SEEDS:
            metrics_path = (
                ROOT
                / dataset
                / "align_sweep"
                / RUN_ID
                / slug
                / f"seed_{seed}"
                / "artifacts"
                / f"alpha_{ALPHA}"
                / "sample_marginal"
                / "metrics.json"
            )
            if not metrics_path.exists():
                continue

            metrics = json.loads(metrics_path.read_text())["interpolation"]
            records.append(
                {
                    "dataset": dataset,
                    "method": method,
                    "alpha": float(ALPHA),
                    "seed": seed,
                    "wasserstein_1": metrics["wasserstein_1"],
                    "wasserstein_2": metrics["wasserstein_2"],
                }
            )

for dataset, artifacts_dir in CELLBRIDGE_ARTIFACTS.items():
    metrics_path = artifacts_dir / f"alpha_{ALPHA}" / "sample_marginal" / "metrics.json"
    if not metrics_path.exists():
        continue

    metrics = json.loads(metrics_path.read_text())["interpolation"]
    records.append(
        {
            "dataset": dataset,
            "method": "CellBridge",
            "alpha": float(ALPHA),
            "seed": "reference",
            "wasserstein_1": metrics["wasserstein_1"],
            "wasserstein_2": metrics["wasserstein_2"],
        }
    )

df = pd.DataFrame(records)
if df.empty:
    raise RuntimeError(f"No alpha=1.0 metrics found for {RUN_ID}")

summary = (
    df.groupby(["dataset", "method", "alpha"], as_index=False)
    .agg(
        n_seeds=("seed", "nunique"),
        wasserstein_1_mean=("wasserstein_1", "mean"),
        wasserstein_1_std=("wasserstein_1", "std"),
        wasserstein_2_mean=("wasserstein_2", "mean"),
        wasserstein_2_std=("wasserstein_2", "std"),
    )
    .fillna(0.0)
)

dataset_order = {"cancer": 0, "immune": 1, "light": 2}
method_order = {"CellBridge": 0, "Shuffle": 1, "Random": 2, "Metacell": 3}
summary = summary.sort_values(
    ["dataset", "method"],
    key=lambda s: s.map(dataset_order) if s.name == "dataset" else s.map(method_order),
).reset_index(drop=True)

display_df = summary.copy()
display_df["dataset"] = display_df["dataset"].replace(
    {"cancer": "Tumor", "immune": "Dendritic", "light": "Light"}
)
display_df["wasserstein_1 (mean±std)"] = display_df.apply(
    lambda r: f"{r['wasserstein_1_mean']:.6f} ± {r['wasserstein_1_std']:.6f}", axis=1
)
display_df["wasserstein_2 (mean±std)"] = display_df.apply(
    lambda r: f"{r['wasserstein_2_mean']:.6f} ± {r['wasserstein_2_std']:.6f}", axis=1
)

display_df[
    [
        "dataset",
        "method",
        "alpha",
        "n_seeds",
        "wasserstein_1 (mean±std)",
        "wasserstein_2 (mean±std)",
    ]
]


,dataset,method,alpha,n_seeds,wasserstein_1 (mean±std),wasserstein_2 (mean±std)
0,Tumor,CellBridge,1.0,1,2.027587 ± 0.000000,2.297803 ± 0.000000
1,Tumor,Shuffle,1.0,3,2.187865 ± 0.007130,2.391820 ± 0.006912
2,Tumor,Random,1.0,3,2.170520 ± 0.029493,2.387680 ± 0.033168
3,Tumor,Metacell,1.0,3,2.051857 ± 0.002321,2.344108 ± 0.001777
4,Dendritic,CellBridge,1.0,1,3.585042 ± 0.000000,3.731680 ± 0.000000
5,Dendritic,Shuffle,1.0,3,3.643913 ± 0.003923,3.747836 ± 0.007320
6,Dendritic,Random,1.0,3,3.601616 ± 0.022050,3.729291 ± 0.019184
7,Dendritic,Metacell,1.0,3,3.587020 ± 0.004700,3.724700 ± 0.000134
8,Light,CellBridge,1.0,1,2.349600 ± 0.000000,2.587260 ± 0.000000
9,Light,Shuffle,1.0,3,2.440959 ± 0.002015,2.645788 ± 0.002328
